# Assignment 2 · Diagnose and improve generated images
**GAN track · Notebook v6 · Chonnam National University · Fall 2026**

Choose **VAE OR GAN**, not both. / VAE 또는 GAN 중 **하나만** 선택하세요. Released 15 October 2026 · Due 4 November 2026, 23:59 KST · 12 points · 10% of course grade.

Inspect generated MNIST images, make one planned change, and judge its result. Both tracks train twice from the same initial seed, use a fixed `z` panel for comparison, and ask for three short pieces of training or generation code. Improved images are **not required** for full credit. / 생성 이미지의 문제를 찾고 변경 계획을 먼저 쓴 뒤 실험하세요. 결과가 개선되지 않아도 근거 있는 분석이면 만점을 받을 수 있습니다.

GAN: the baseline G learning rate is 0.00004; the intervention is 0.0002. D learning rate stays 0.0002. Both use non-saturating G loss. This is an image GAN, not the old eight-cluster ring experiment.

Active work target: **3–5 hours** including a 16-image audit and 1–2-page analysis. Two CPU training runs only; no hyperparameter search. Submit the executed notebook and **one PDF** with your 1–2-page analysis followed by the two-page receipt. See `A2_Student_Assignment_v7.pdf`. English or Korean answers are accepted; technical terms can remain in English.

## 0 Setup and data / 준비
Run these supplied cells. `auto` downloads MNIST if needed. If download fails, an offline sklearn digits dataset runs as a smoke test, but its result is **not submission-ready**. No Drive mount, GPU, or paid service is required. Reference code is openly provided through `check_code(N, use_reference=True)` and has **no penalty**; explain each block in your report.

In [ ]:
#@title Setup and workflow / 준비 및 실행 관리 {display-mode:"form"}
import copy, gzip, hashlib, json, math, os, platform, struct, time, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display, FileLink

torch.set_num_threads(2)
VERSION = 'A2 redesign v7 / notebook v6'
SEED = 0
BASE, VARIANT, PLAN, AUDIT = None, None, None, None
CODE_STATUS = {}
OUT = Path('a2_outputs'); OUT.mkdir(exist_ok=True)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()[:16]

def array_id(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()[:16]

def code_id():
    # Detect ordinary edits after a run; this is not an anti-tampering system.
    return fingerprint({name: (globals()[name].__code__.co_code.hex(), repr(globals()[name].__code__.co_consts)) for name in CODE_NAMES})

def check_code(number, use_reference=False):
    name = CODE_NAMES[number - 1]
    if use_reference:
        globals()[name] = REFERENCES[number - 1]
    try:
        test_student_code(number)
        mode = 'reference' if globals()[name] is REFERENCES[number - 1] else 'student'
        CODE_STATUS[name] = mode
        print(f'Code {number}: PASS ({mode})')
        return True
    except Exception as exc:
        CODE_STATUS[name] = 'pending'
        print(f'Code {number}: NOT READY. {type(exc).__name__}: {exc}')
        print('update, tensor shape, eval/no_grad를 확인하세요. 필요하면 check_code(number, use_reference=True)를 사용하세요. 감점은 없습니다.')
        return False

def record_plan(student_id, diagnosis, candidate, prediction, success_rule):
    global PLAN, VARIANT, AUDIT
    if BASE is None:
        print('먼저 baseline을 실행하세요 / Run the baseline first.'); return
    values = [student_id, diagnosis, prediction, success_rule]
    if any(not isinstance(x, str) or not x.strip() for x in values):
        print('NOT READY: 학번, baseline 관찰, 예측, 판단 기준을 작성하세요.'); return
    if candidate not in OPTIONS:
        print('NOT READY: 이 track에서 허용된 변경값을 사용하세요 / use the assigned option.'); return
    draft = dict(student_id=student_id.strip(), diagnosis=diagnosis.strip(), candidate=float(candidate),
                 prediction=prediction.strip(), success_rule=success_rule.strip(), baseline_id=BASE['id'])
    if PLAN != draft:
        PLAN = draft; VARIANT = None; AUDIT = None
    print('PLAN SAVED. 변경 실험을 실행하세요 / Run your chosen intervention.')

def run_baseline():
    global BASE, VARIANT, PLAN, AUDIT
    cfg = config(BASE_VALUE)
    if BASE is not None and BASE['config'] == cfg:
        print('기존 baseline 재사용 / Reusing baseline.'); return
    BASE = train(BASE_VALUE, student=False)
    PLAN = VARIANT = AUDIT = None

def run_variant():
    global VARIANT, AUDIT
    if PLAN is None or BASE is None:
        print('NOT READY: baseline을 보고 변경 계획을 먼저 작성하세요.'); return
    if BASE['config'] != config(BASE_VALUE) or PLAN['baseline_id'] != BASE['id']:
        print('NOT READY: 설정이 바뀌었습니다. baseline부터 다시 실행하세요.'); return
    if not all([check_code(i) for i in [1, 2, 3]]):
        print('NOT READY: 코드 세 칸을 완성하거나 제공 코드로 전환하세요.'); return
    key = fingerprint(PLAN)
    if VARIANT is not None and VARIANT['plan_id'] == key and VARIANT['code_id'] == code_id():
        print('기존 변경 실험 재사용 / Reusing intervention run.'); return
    VARIANT = train(PLAN['candidate'], student=True)
    VARIANT.update(plan_id=key, code_id=code_id())
    AUDIT = None

def ready_reason():
    if BASE is None or PLAN is None or VARIANT is None:
        return 'baseline, plan 또는 intervention 실행이 남아 있습니다.'
    if BASE['config'] != config(BASE_VALUE) or VARIANT['config'] != config(PLAN['candidate']):
        return '설정이 바뀌었습니다. baseline부터 다시 실행하세요.'
    if VARIANT['plan_id'] != fingerprint(PLAN) or PLAN['baseline_id'] != BASE['id']:
        return '변경 계획과 실행 결과가 다릅니다.'
    if VARIANT['code_id'] != code_id() or any(CODE_STATUS.get(n) == 'pending' for n in CODE_NAMES):
        return '코드가 바뀌었습니다. 코드 점검과 변경 실험을 다시 실행하세요.'
    return ''

def make_receipt(ai_use):
    reason = ready_reason()
    if not reason and 'OFFLINE' in DATA_NAME:
        reason = '오프라인 smoke test입니다. 제출 전 MNIST로 두 run을 실행하세요.'
    if not reason and (not isinstance(ai_use, str) or not ai_use.strip()):
        reason = 'AI 사용 내용을 쓰세요. 사용하지 않았다면 none을 적으세요.'
    if not reason and (AUDIT is None or AUDIT['run_id'] != VARIANT['id']):
        reason = '두 grid의 sample 01–16에 대한 visual audit를 작성하세요.'
    if reason:
        print('NOT READY:', reason); return None
    b, v = metrics(BASE), metrics(VARIANT)
    payload = dict(version=VERSION, track=TRACK, plan=PLAN, code=CODE_STATUS.copy(),
                   baseline=dict(config=BASE['config'], metrics=b, sample_id=array_id(BASE['samples'])),
                   intervention=dict(config=VARIANT['config'], metrics=v, sample_id=array_id(VARIANT['samples'])),
                   visual_audit=AUDIT, ai_use=ai_use.strip(),
                   environment=dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__, device='cpu'))
    fid=fingerprint(payload)
    receipt=dict(status='READY', fingerprint=fid, **payload)
    (OUT/f'A2_{TRACK}_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf8')
    # English labels and numbers in the PDF avoid missing Korean fonts on Colab.
    # The full original Korean plan/AI statement is preserved in the notebook and JSON.
    lines=[f'{VERSION} | {TRACK} | READY',f'Fingerprint: {fid}',
           f'Data: {BASE["config"]["data"]}', f'Changed parameter: {PARAMETER}',
           f'Baseline: {BASE_VALUE}   Intervention: {PLAN["candidate"]}',
           'Only the named parameter changes; both runs restart from the same seed.',
           'Settings: '+json.dumps(BASE['config'],ensure_ascii=True), '',
           'BASELINE', *[f'  {k}: {v}' for k,v in b.items()], '',
           'INTERVENTION', *[f'  {k}: {v}' for k,v in v.items()], '',
           'Code: '+str(CODE_STATUS),
           'Full plan, AI statement and annotations: notebook and receipt JSON.',
           'This receipt is an appendix. Write your own analysis in the report.',
           'Fingerprint checks consistency; it does not prove authorship.']
    import textwrap
    wrapped='\n'.join(y for x in lines for y in (textwrap.wrap(x,95) or ['']))
    pdf_path=OUT/f'A2_{TRACK}_receipt.pdf'
    with PdfPages(pdf_path) as pdf:
        fig=plt.figure(figsize=(8.27,11.69));fig.text(.07,.95,wrapped,va='top',fontfamily='monospace',fontsize=8.5);pdf.savefig(fig);plt.close(fig)
        fig=comparison_figure();pdf.savefig(fig,bbox_inches='tight');plt.close(fig)
    print('READY | fingerprint:',fid)
    print('Baseline:',b); print('Intervention:',v)
    print('분석 보고서 뒤에 PDF를 붙이세요. JSON은 노트북과 함께 보관하면 됩니다.')
    display(FileLink(str(pdf_path)));display(FileLink(str(OUT/f'A2_{TRACK}_receipt.json')))
    return receipt

In [ ]:
#@title Model, data, training and figures / model, data, 학습 및 그림 {display-mode:"form"}
TRACK='GAN'
PARAMETER='generator learning rate'
BASE_VALUE=0.00004
OPTIONS=(0.0002,)
EPOCHS=3; BATCH=128; Z_DIM=16; D_LR=0.0002
DATA_SOURCE='auto';DATA_NAME=None
Z_FIXED=torch.randn(64,Z_DIM,generator=torch.Generator().manual_seed(7))

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc=nn.Sequential(nn.Linear(Z_DIM,128*7*7),nn.BatchNorm1d(128*7*7),nn.ReLU())
        self.up=nn.Sequential(nn.ConvTranspose2d(128,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(),nn.ConvTranspose2d(64,1,4,2,1),nn.Tanh())
    def forward(self,z):return self.up(self.fc(z).view(-1,128,7,7))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(1,64,4,2,1),nn.LeakyReLU(.2),nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(.2),nn.Flatten(),nn.Linear(128*7*7,1))
    def forward(self,x):return self.net(x)

def reference_d_update(D,opt,real,fake):
    real_logits=D(real);fake_logits=D(fake.detach())
    loss=F.binary_cross_entropy_with_logits(real_logits,torch.ones_like(real_logits))+F.binary_cross_entropy_with_logits(fake_logits,torch.zeros_like(fake_logits))
    opt.zero_grad();loss.backward();opt.step()
    return float(loss.detach())

def reference_g_update(D,opt,fake):
    fake_logits=D(fake)
    loss=F.binary_cross_entropy_with_logits(fake_logits,torch.ones_like(fake_logits))
    opt.zero_grad();loss.backward();opt.step()
    return float(loss.detach())

def reference_generate_images(G,z):
    was_training=G.training;G.eval()
    with torch.no_grad(): out=((G(z)+1)/2).clamp(0,1).cpu().numpy()
    G.train(was_training)
    return out

CODE_NAMES=['d_update','g_update','generate_images']
REFERENCES=[reference_d_update,reference_g_update,reference_generate_images]

def test_student_code(number):
    fn=globals()[CODE_NAMES[number-1]]
    torch.manual_seed(709)
    if number==1:
        a=Discriminator();b=copy.deepcopy(a)
        oa=torch.optim.SGD(a.parameters(),lr=.001);ob=torch.optim.SGD(b.parameters(),lr=.001)
        real=torch.randn(3,1,28,28);fake=torch.randn(3,1,28,28,requires_grad=True)
        expected=reference_d_update(a,oa,real,fake)
        actual=fn(b,ob,real,fake)
        assert actual is not None and abs(actual-expected)<1e-5, 'D loss differs'
        assert fake.grad is None, 'D update must detach fake images'
        assert all(torch.allclose(x,y,atol=1e-5,rtol=1e-4) for x,y in zip(a.parameters(),b.parameters())), 'D optimizer update differs'
    elif number==2:
        g1=Generator();g2=copy.deepcopy(g1);d=Discriminator()
        o1=torch.optim.SGD(g1.parameters(),lr=.001);o2=torch.optim.SGD(g2.parameters(),lr=.001)
        z=torch.randn(3,Z_DIM)
        f1=g1(z);f2=g2(z)
        expected=reference_g_update(d,o1,f1)
        actual=fn(d,o2,f2)
        assert actual is not None and abs(actual-expected)<1e-5, 'G loss differs'
        assert all(torch.allclose(x,y,atol=1e-5,rtol=1e-4) for x,y in zip(g1.parameters(),g2.parameters())), 'G optimizer update differs; keep the gradient path through D'
    else:
        g=Generator();g.train();z=torch.randn(4,Z_DIM)
        expected=reference_generate_images(g,z)
        actual=fn(g,z)
        assert actual is not None and isinstance(actual,np.ndarray), 'return NumPy images'
        assert actual.shape==expected.shape and np.allclose(actual,expected,atol=1e-6), 'images differ; use eval mode and map [-1,1] to [0,1]'
        assert g.training, 'restore the original training mode'

def load_data(source='auto'):
    global X_TRAIN,X_TEST,DATA_NAME,DATA_SOURCE
    DATA_SOURCE=source
    if source not in ['auto','offline']:raise ValueError('source must be auto or offline')
    try:
        if source=='offline':raise OSError('offline mode requested')
        arrays=[]
        for name in ['train-images-idx3-ubyte.gz','t10k-images-idx3-ubyte.gz']:
            p=Path(name)
            if not p.exists():
                last=None
                for base in ['https://ossci-datasets.s3.amazonaws.com/mnist/','https://storage.googleapis.com/cvdf-datasets/mnist/']:
                    try:
                        with urllib.request.urlopen(base+name,timeout=8) as response:raw=response.read()
                        decoded=gzip.decompress(raw);magic,n,r,c=struct.unpack('>IIII',decoded[:16])
                        assert magic==2051 and r==28 and c==28 and len(decoded)==16+n*r*c
                        p.write_bytes(raw);break
                    except Exception as exc:last=exc
                else:raise OSError(f'MNIST download failed: {last}')
            with gzip.open(p,'rb') as f:
                magic,n,r,c=struct.unpack('>IIII',f.read(16));assert magic==2051 and r*c==784
                arrays.append(torch.from_numpy(np.frombuffer(f.read(),np.uint8).copy().reshape(n,1,28,28)).float()/127.5-1)
        X_TRAIN,X_TEST=arrays[0][:20000],arrays[1]
        DATA_NAME='MNIST first 20000 train / 10000 test'
    except (OSError,AssertionError,ValueError,EOFError,struct.error) as exc:
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
        ds=load_digits();x=torch.tensor(ds.images/16,dtype=torch.float32)[:,None]
        x=F.interpolate(x,size=(28,28),mode='bilinear',align_corners=False)*2-1
        tr,te=train_test_split(np.arange(len(x)),test_size=.2,random_state=42,stratify=ds.target)
        X_TRAIN,X_TEST=x[tr],x[te];DATA_NAME='OFFLINE sklearn digits SMOKE TEST ONLY'
        print('MNIST unavailable:',type(exc).__name__)
        print('오프라인 실행 점검용 data입니다. 제출용 MNIST와 다르며 READY PDF를 만들 수 없습니다.')
    print(DATA_NAME,tuple(X_TRAIN.shape),tuple(X_TEST.shape))

def config(value):
    return dict(data=DATA_NAME,seed=SEED,epochs=EPOCHS,batch=BATCH,z_dim=Z_DIM,g_lr=float(value),d_lr=D_LR,
                train_size=len(X_TRAIN),loss='non-saturating',optimizer='Adam betas=(0.5,0.999)',
                updates='D1 G1',fixed_z_seed=7)

def train(value,student=False):
    start=time.time();torch.manual_seed(SEED)
    G,D=Generator(),Discriminator()
    og=torch.optim.Adam(G.parameters(),lr=value,betas=(.5,.999))
    od=torch.optim.Adam(D.parameters(),lr=D_LR,betas=(.5,.999))
    history=[];snaps={}
    for ep in range(1,EPOCHS+1):
        perm=torch.randperm(len(X_TRAIN));ld=lg=dr=df=0.;n=0
        for i in range(0,len(X_TRAIN),BATCH):
            real=X_TRAIN[perm[i:i+BATCH]]
            fake=G(torch.randn(len(real),Z_DIM))
            dloss=(d_update if student else reference_d_update)(D,od,real,fake)
            gloss=(g_update if student else reference_g_update)(D,og,fake)
            with torch.no_grad():
                dr+=float(D(real).sigmoid().mean());df+=float(D(fake.detach()).sigmoid().mean())
            ld+=dloss;lg+=gloss;n+=1
        snaps[ep]=(generate_images if student else reference_generate_images)(G,Z_FIXED)
        history.append([ep,ld/n,lg/n,dr/n,df/n])
        print(f'epoch {ep}/{EPOCHS} | D loss {ld/n:.3f} | G loss {lg/n:.3f} | D(real) {dr/n:.3f} | D(fake) {df/n:.3f}')
    samples=snaps[EPOCHS]
    assert np.isfinite(samples).all()
    result=dict(config=config(value),samples=samples,snapshots=snaps,history=history,seconds=round(time.time()-start,2))
    result['id']=fingerprint([result['config'],array_id(samples)])
    print('Completed in',result['seconds'],'s. G lr =',value)
    return result

def _grid(fig,slot,images,rows=4):
    sub=slot.subgridspec(rows,8,wspace=.02,hspace=.08)
    for i in range(rows*8):
        ax=fig.add_subplot(sub[i//8,i%8]);ax.imshow(images[i].reshape(28,28),cmap='gray',vmin=0,vmax=1);ax.axis('off')
        if i<16:ax.set_title(str(i+1).zfill(2),fontsize=7,pad=0)

def baseline_figure():
    if BASE is None:return
    fig=plt.figure(figsize=(10,5));gs=fig.add_gridspec(1,2)
    _grid(fig,gs[0],((X_TEST[:32]+1)/2).numpy())
    _grid(fig,gs[1],BASE['samples'])
    fig.text(.26,.96,'Real MNIST',ha='center');fig.text(.75,.96,'Baseline generated | fixed z',ha='center')
    plt.show();plt.close(fig)

def comparison_figure():
    fig=plt.figure(figsize=(11,12));gs=fig.add_gridspec(3,2,hspace=.28)
    _grid(fig,gs[0,0],((X_TEST[:32]+1)/2).numpy())
    _grid(fig,gs[0,1],BASE['samples'])
    _grid(fig,gs[1,0],VARIANT['samples'])
    _grid(fig,gs[1,1],BASE['snapshots'][1])
    _grid(fig,gs[2,0],VARIANT['snapshots'][1])
    _grid(fig,gs[2,1],VARIANT['snapshots'][2])
    for x,y,t in [(.28,.94,'Real MNIST'),(.74,.94,'Baseline epoch 3'),(.27,.64,'Intervention epoch 3'),(.74,.64,'Baseline epoch 1'),(.27,.33,'Intervention epoch 1'),(.74,.33,'Intervention epoch 2')]:fig.text(x,y,t,ha='center',fontsize=10)
    fig.suptitle('Same fixed z within each run | image formation across epochs',y=.99)
    return fig

def save_audit(baseline_labels,intervention_labels):
    global AUDIT
    if ready_reason():print('NOT READY: 먼저 변경 실험을 완료하세요.');return
    parsed=[]
    for item in [baseline_labels,intervention_labels]:
        labels=[x.strip().lower() for x in item.split(',')]
        if len(labels)!=16 or any(x not in list('0123456789')+['x'] for x in labels):
            AUDIT=None;print('NOT READY: 01–16 labels must be 0–9 or x, separated by commas.');return
        parsed.append(labels)
    AUDIT=dict(run_id=VARIANT['id'],baseline=parsed[0],intervention=parsed[1])
    print('Visual audit saved. These 16 subjective labels are not an accuracy or mode-collapse score.')

def metrics(run):
    h=run['history'][-1]
    out=dict(final_D_loss=round(h[1],3),final_G_loss=round(h[2],3),D_real=round(h[3],3),D_fake=round(h[4],3))
    if AUDIT is not None and VARIANT is not None and AUDIT['run_id']==VARIANT['id']:
        labels=AUDIT['baseline' if run is BASE else 'intervention']
        out.update(audited_samples=16,readable_share=round(float(np.mean(np.asarray(labels)!='x')),4),
                   distinct_readable_digits=len(set(labels)-{'x'}),labels=labels)
    return out

def show_comparison():
    reason=ready_reason()
    if reason:print('NOT READY:',reason);return
    fig=comparison_figure();fig.savefig(OUT/'GAN_comparison.png',dpi=120);plt.show();plt.close(fig)
    print('Baseline',metrics(BASE));print('Intervention',metrics(VARIANT))
    print('Label generated samples 01–16 in the top baseline and middle-left intervention panels. Use x when unclear.')

def optional_loss_plot():
    if ready_reason():return
    fig,axes=plt.subplots(1,4,figsize=(12,3))
    for run,label in [(BASE,'Baseline'),(VARIANT,'Intervention')]:
        h=np.array(run['history'])
        for j in range(4):axes[j].plot(h[:,0],h[:,j+1],marker='o',label=label)
    for a,t in zip(axes,['D loss','G loss','D(real)','D(fake)']):a.set(xlabel='Epoch',title=t);a.legend(fontsize=7)
    fig.tight_layout();plt.show();plt.close(fig)

In [ ]:
#@title MNIST data / MNIST 준비
load_data('auto')
# To check execution without internet: load_data('offline'). Restore auto for submission.

## 1 Observe baseline images / Baseline 관찰
Run the baseline and compare **real MNIST** with **prior/generated** images. Name a visible limitation using sample IDs 01–16. Do not diagnose a learning mechanism from loss alone. / 먼저 실제 이미지와 생성 이미지를 보고 sample 번호로 문제를 짚으세요.

In [ ]:
#@title Baseline training and samples / Baseline 학습과 생성 이미지
run_baseline()
baseline_figure()

## 2 Record the intervention plan / 변경 전 계획
Change **G learning rate 0.00004 → 0.0002**. D learning rate, model, seed, 20,000 training images, batch order, noise sequence, three epochs and fixed evaluation `z` stay the same. The two runs restart from the same weights; the intervention is not a continuation. A larger learning rate can strengthen G updates, but can also destabilize training. State a visual prediction **before** running it. / 학습과 생성 결과를 보기 전에 예상되는 개선과 확인할 부작용을 적으세요.

In [ ]:
#@title Record the plan before training / 변경 전 계획
student_id = '' #@param {type:"string"}
diagnosis = '' #@param {type:"string"}
new_value = 0.0002  # prescribed GAN intervention
prediction = '' #@param {type:"string"}
success_rule = '' #@param {type:"string"}
record_plan(student_id, diagnosis, new_value, prediction, success_rule)

## 3 Complete three useful code blocks / 코드 세 칸
These blocks drive the **intervention** training and generated images. The baseline uses the same supplied reference operations, so the comparison changes only the planned hyperparameter when your code passes its self-check. Small checks verify loss, optimizer changes, gradient path, and output format; they do not reveal your experimental answer. / Baseline은 같은 reference 동작으로 실행됩니다. 코드가 막히면 `check_code(1, use_reference=True)`처럼 전환하세요. 감점은 없고, 보고서에는 계산과 역할을 설명해야 합니다. SyntaxError가 나면 답안 줄을 `None`으로 되돌린 다음 전환하세요.

### Code 1 · D optimizer update
Use `D(real)` and `D(fake.detach())`, binary cross-entropy with logits and real/fake targets, then zero gradients, backpropagate and step. Return the scalar D loss. / D update에서 `fake.detach()`가 왜 필요한지도 설명하세요.

In [ ]:
def d_update(D, opt, real, fake):
    return None  # TODO: replace this block with working code

check_code(1)

### Code 2 · G optimizer update
Use D on `fake` **without detach** and a real target. Step the G optimizer and return scalar G loss. / D를 통과하는 gradient가 G에 닿아야 합니다.

In [ ]:
def g_update(D, opt, fake):
    return None  # TODO: replace this block with working code

check_code(2)

### Code 3 · Generate fixed-z images
Switch to `eval`, use `torch.no_grad`, map tanh output from [-1,1] to [0,1], return a NumPy array, then restore the prior mode. / BatchNorm이 있는 G의 평가 모드와 밝기 범위를 확인하세요.

In [ ]:
def generate_images(G, z):
    return None  # TODO: replace this block with working code

check_code(3)

In [ ]:
#@title Run the one planned intervention / 변경 실험 한 번 실행
run_variant()

## 4 Compare image formation / 생성 이미지 비교
Use the same fixed `z` IDs. The figure shows **real MNIST**, final baseline/intervention images, and epoch-1/2 checkpoints. A GAN can produce repeated recognizable images (limited variety), only noise (insufficient learning), or both; these are different observations. `D(real)` and `D(fake)` help explain training dynamics but do not measure image quality or prove mode collapse. / 같은 `z`의 이미지가 epoch별로 어떻게 형성되는지 보세요. 알아보기 어려운 것과 알아볼 수 있지만 반복되는 것을 구분하세요.

Why might concentration arise? G receives feedback on **its generated samples**. If some outputs start to receive useful gradients earlier, updates can move many `z` values toward a subset of patterns. D then adapts; the outcome depends on their balance. A 16-image panel alone cannot establish this cause or full mode coverage. / 이 설명은 가능한 mechanism입니다. 이번 실험이 이를 증명하는 것은 아닙니다.

In [ ]:
#@title Compare generated images / 생성 이미지 비교
show_comparison()

### Visual audit of 16 generated images / 작은 visual audit
For each generated grid, label images **01–16** with one digit 0–9 if clearly recognizable; otherwise `x`. Use the same rule in both runs. `readable share` and `distinct readable digits` summarize only this small subjective sample, not full distribution coverage or classifier accuracy. Cite at least two image IDs in your report. / 두 run에 같은 기준을 적용하세요. 애매하면 x입니다. 숫자가 적다고 mode collapse가 입증되는 것은 아닙니다.

In [ ]:
#@title Record image IDs 01–16 / 생성 이미지 판정
baseline_labels = '' #@param {type:"string"}
intervention_labels = '' #@param {type:"string"}
save_audit(baseline_labels, intervention_labels)

Optional: inspect training curves only to explain observations. G and D losses have different meanings and do not rank image quality. / Loss curve만으로 생성 성능을 판단하지 마세요.

In [ ]:
#@title Optional curves / 선택 확인
show_curves = False #@param {type:"boolean"}
if show_curves:
    optional_loss_plot()

## 5 Short analysis / 분석 본문
Write **1–2 pages** and then attach the automatic **two-page receipt**. Answer in English or Korean. / 본문 1–2쪽 뒤에 결과 기록 2쪽을 붙여 PDF 하나로 제출하세요.

**Q1 Baseline diagnosis (2 points).** State a visible generation problem, supported by two specific image IDs or features. For VAE, separate prior generation from reconstruction; for GAN, separate unreadable images from repeated readable patterns.

**Q2 Planned change (2 points).** Explain the hyperparameter, your prediction, what stayed fixed, and one possible cost. Do not rewrite your original prediction.

**Q3 What changed (4 points).** Give a verdict based on a direct image comparison, the 16-image audit (readable share **and** distinct digits), and one supporting measurement or checkpoint. Describe at least one remaining problem and distinguish observed facts from a proposed cause. A negative or mixed result may receive full marks.

**Q4 Next test and project bridge (2 points).** Propose one next controlled test that could distinguish an unresolved explanation and name the result that would change your mind. Suggest one final-project dataset/task and a matching generation check. No further run or project commitment is required.

**Code and reproducibility (2 points).** Each block: 0.25 for passing execution + 0.25 for a correct explanation of its training/generation role in your report; 0.5 for a READY receipt consistent with the executed notebook. Reference code is accepted without penalty. A numeric typo is deducted once, in the relevant question only.

## 6 Export receipt / 결과 기록 저장
Write an AI-use statement (`none` if unused). The receipt has settings, audit, metrics, image comparison and a fingerprint. It is an appendix; write your own analysis. Before submission run all cells again, review images and receipt, and save outputs. A cache is reused only while settings and code match. / PDF는 `a2_outputs`에 저장됩니다. Colab 왼쪽 Files에서 받으세요. READY는 자료가 갖춰졌다는 뜻이지 분석의 정답 판정이 아닙니다.

In [ ]:
#@title Export the two-page receipt / 결과 기록 PDF
ai_use = '' #@param {type:"string"}
receipt = make_receipt(ai_use)